# Inverse Kinematics

**Primary Reference:** Lynch & Park, *Modern Robotics: Mechanics, Planning, and Control*, Ch. 6  
Northwestern University — freely available at http://modernrobotics.org  

**Supporting References:**  
- Craig, *Introduction to Robotics: Mechanics and Control*, Ch. 4 (Analytical IK)  
- MIT 6.832 Underactuated Robotics — https://underactuated.mit.edu  
- CMU 16-311 Introduction to Robotics, Lecture 8 — https://www.cs.cmu.edu/~16311  
- Spong, Hutchinson & Vidyasagar, *Robot Modeling and Control*, Ch. 4  

---

## What is Inverse Kinematics?

Forward kinematics answers: *given joint angles, where is the end-effector?*  
Inverse kinematics (IK) answers the opposite question: *given a desired end-effector position, what joint angles do I need?*

**Analogy:** Imagine reaching for a coffee cup on your desk. Your brain doesn't think about shoulder angle and elbow angle — it thinks about where the cup is, and your nervous system figures out the joint angles automatically. That "figuring out" is inverse kinematics.

Mathematically, forward kinematics gave us a function:

$$
T = f(\theta_1, \theta_2, \dots, \theta_n)
$$

Inverse kinematics asks: given a desired $T$, find $\theta_1, \theta_2, \dots, \theta_n$.

This sounds simple — just invert the function. But in practice it is much harder than forward kinematics for three reasons:

1. **Multiple solutions** — there may be many joint configurations that reach the same point (think of your elbow up vs. elbow down when reaching forward)
2. **No solution** — the target may be outside the robot's workspace
3. **No closed form** — for most robots, you cannot write a clean algebraic formula for the answer

---

## Two Approaches to IK

### 1. Analytical (Closed-Form)

For robots with special geometry (e.g. three consecutive joint axes intersecting at a point — called a **spherical wrist**), it is possible to derive exact algebraic formulas for the joint angles.

- **Advantage:** Fast, exact, gives all solutions explicitly  
- **Disadvantage:** Only works for specific robot geometries; requires significant hand derivation  
- **Reference:** Craig Ch. 4 works through the 6-DOF spherical wrist case in detail

### 2. Numerical (Iterative)

For general robots, we use iteration: start with a guess for the joint angles, compute where the end-effector ends up, measure the error, and adjust the joint angles to reduce that error. Repeat until close enough.

- **Advantage:** Works for any robot geometry, straightforward to implement  
- **Disadvantage:** May converge slowly, may get stuck at a local minimum, does not guarantee finding all solutions  
- **Reference:** Lynch & Park Ch. 6.2 covers Newton-Raphson IK in detail

This project will use the **numerical approach**, built around the Jacobian matrix.

---

## The Jacobian — Intuition First

Before we can do numerical IK, we need a tool called the **Jacobian**.

**Intuition:** The Jacobian tells you how a small change in joint angles produces a small change in end-effector position. It is the "sensitivity" of the end-effector to each joint.

Think of it like a gear ratio: if you turn joint 2 by a tiny amount, the end-effector moves by some amount in x, y, and z. The Jacobian captures all of those relationships at once.

**Formally:** If $\mathbf{x}$ is the end-effector position and $\boldsymbol{\theta}$ is the vector of joint angles, the Jacobian $J$ relates their velocities:

$$
\dot{\mathbf{x}} = J(\boldsymbol{\theta}) \, \dot{\boldsymbol{\theta}}
$$

Where:
- $\dot{\mathbf{x}}$ is the end-effector velocity (6×1 for position + orientation)
- $\dot{\boldsymbol{\theta}}$ is the vector of joint velocities (n×1)
- $J$ is a 6×n matrix — each column describes how one joint moves the end-effector

**Reference:** Lynch & Park Ch. 5.1 — Manipulator Jacobian  
**Reference:** CMU 16-311 Lecture 9 — Jacobian and Velocity Kinematics

---

## Building the Jacobian from the T Matrices

For a serial robot, each column $j$ of the Jacobian comes from the transformation matrix up to joint $j$.

For a **revolute joint** $j$, the column of $J$ is:

$$
J_j = \begin{bmatrix} z_{j-1} \times (p_n - p_{j-1}) \\ z_{j-1} \end{bmatrix}
$$

Where:
- $z_{j-1}$ is the z-axis of frame $j-1$ (third column of the rotation part of $^0T_{j-1}$)
- $p_n$ is the end-effector position (top-right column of $^0T_n$)
- $p_{j-1}$ is the origin of frame $j-1$ (top-right column of $^0T_{j-1}$)
- $\times$ is the cross product

For a **prismatic joint** $j$:

$$
J_j = \begin{bmatrix} z_{j-1} \\ 0 \end{bmatrix}
$$

This is why knowing the `type` of each joint matters — revolute and prismatic joints contribute differently to the Jacobian.

**Reference:** Spong Ch. 4.10 — The Geometric Jacobian

---

## Numerical IK — Newton-Raphson Method

With the Jacobian in hand, the iterative IK algorithm works as follows:

1. Start with an initial guess $\boldsymbol{\theta}^{(0)}$
2. Compute the current end-effector pose: $T = f(\boldsymbol{\theta}^{(k)})$
3. Compute the error between current and desired pose: $\mathbf{e}^{(k)}$
4. Compute the Jacobian at the current configuration: $J(\boldsymbol{\theta}^{(k)})$
5. Update joint angles:

$$
\boldsymbol{\theta}^{(k+1)} = \boldsymbol{\theta}^{(k)} + J^{\dagger} \, \mathbf{e}^{(k)}
$$

6. Repeat until $\|\mathbf{e}\| < \epsilon$ (error is small enough)

Where $J^{\dagger}$ is the **Moore-Penrose pseudoinverse** of the Jacobian: `np.linalg.pinv(J)`

The pseudoinverse is used instead of a regular inverse because $J$ is generally not square (6×n where n may not equal 6).

**Reference:** Lynch & Park Ch. 6.2 — Numerical Inverse Kinematics  
**Reference:** MIT 6.832 Notes on Optimization and IK — https://underactuated.mit.edu

---

## Singularities

A **singularity** occurs when the robot reaches a configuration where it loses the ability to move in certain directions, no matter which joints you move.

**Analogy:** Fully extend your arm straight out in front of you. You can no longer move your hand further forward by rotating your elbow — you've lost one degree of freedom. That is a singular configuration.

Mathematically, a singularity is where $J$ loses rank — i.e. $\det(J) = 0$ (or near zero). The pseudoinverse blows up near singularities, which causes numerical IK to behave erratically.

**How to detect:** Monitor $\det(J J^T)$ during iteration. If it approaches zero, the robot is near a singularity.

**Reference:** Spong Ch. 4.11 — Singularities  
**Reference:** CMU 16-311 Lecture 9

---

## Workspace

The **workspace** is the set of all positions the end-effector can reach.

- **Reachable workspace** — all positions the end-effector can reach in at least one orientation
- **Dexterous workspace** — positions the end-effector can reach in *any* orientation

IK has no solution outside the reachable workspace. This is important for motor selection — the workspace is determined by link lengths and joint limits, which are design choices you control.

**Reference:** Craig Ch. 4.1 — Workspace

---

## What to Implement

Based on this project's structure, IK will require:

1. **`jacobian.py`** — builds the geometric Jacobian from the list of T matrices and joint types
2. **`inverse_kinematics.py`** — Newton-Raphson iteration using the Jacobian pseudoinverse
3. The controller passes the desired end-effector pose and initial joint angle guess; IK returns the joint angles

The Jacobian also reappears in dynamics (it maps joint torques to end-effector forces), so it deserves its own file.

## Example — Computing One Column of the Jacobian

In [ ]:
import numpy as np

# Suppose T_list is your list of 4x4 transformation matrices from transforms.py
# T_list[i] is ^0T_{i+1} (0-indexed)

# For joint j (revolute), the Jacobian column is built from ^0T_{j-1}
# z_{j-1} = third column of the rotation block = T_prev[:3, 2]
# p_{j-1} = origin of frame j-1           = T_prev[:3, 3]
# p_n     = end-effector origin            = T_end[:3, 3]

def jacobian_column_revolute(T_prev, T_end):
    z = T_prev[:3, 2]
    p_prev = T_prev[:3, 3]
    p_end = T_end[:3, 3]
    linear = np.cross(z, p_end - p_prev)
    angular = z
    return np.concatenate([linear, angular])  # 6x1 column

def jacobian_column_prismatic(T_prev):
    z = T_prev[:3, 2]
    return np.concatenate([z, np.zeros(3)])   # 6x1 column

## Example — Newton-Raphson IK Iteration (Sketch)

In [ ]:
# This is a conceptual sketch — not the final implementation
# It shows the structure of the Newton-Raphson loop

# np.linalg.pinv computes the Moore-Penrose pseudoinverse
# Reference: numpy docs — https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html

def ik_newton_raphson(target_T, theta_init, forward_kin_fn, jacobian_fn, tol=1e-6, max_iter=100):
    theta = theta_init.copy()

    for _ in range(max_iter):
        current_T = forward_kin_fn(theta)          # current end-effector pose
        error = compute_error(target_T, current_T) # 6x1 position+orientation error

        if np.linalg.norm(error) < tol:
            break

        J = jacobian_fn(theta)                     # 6xn Jacobian at current config
        J_pinv = np.linalg.pinv(J)                 # pseudoinverse
        theta = theta + J_pinv @ error             # Newton-Raphson update step

    return theta

# compute_error needs to measure both position and orientation difference
# For position: target_T[:3,3] - current_T[:3,3]
# For orientation: use the rotation matrix difference or axis-angle representation
# Reference: Lynch & Park Ch. 3.3 — Exponential Coordinates for rotation error